# RawWriter2p: raw ファイルの書き出し

TPCEmulator と HAEmulator が作ったイベントを、**実データと同じ形式の raw ファイル** に書きます。
あわせて、書いたイベントの真の値を `sim_truth.root` に書きます。チェーンの最後のモジュールです。

| | |
| --- | --- |
| SMILE_tool_box | `EmuWrite2p` |
| ソース | `simulation_source/modules/RawWriter2p/`(ファイルの形式は `simulation_source/vendor/emu_write/` の `TPC_FileManager`, `HA_FileManager`) |

## 1. 何をするか

- 初期化: `output_dir` を作り、ボードごと(`uTPC_anode<board>_00000.raw`, `uTPC_cathode<board>_00000.raw`)と HA ごと(`HA<名前>_00000.raw`)にファイルを開く
- イベントごと(SimTrigger を通ったイベントだけ): TPCEmulator のヒットと FADC、HAEmulator の ADC を、イベント番号(`EdepPickUp::EventID()`)と時刻を付けて書く。
  真の値のツリー `sim` に 1 行書く
- 終わり: ファイルを閉じる

イベント番号が全ボードで同じなので、`ReadData` は実データと同じようにボード間でイベントを揃えて読めます。

## 2. パラメータ

| パラメータ | 型 | 既定値(C++) | この設定の値 | 単位 | 説明 |
| --- | --- | --- | --- | --- | --- |
| `output_dir` | string | rawdata | rawdata |  | raw ファイルを書くディレクトリ |
| `truth_file` | string | (空) | rawdata/sim_truth.root |  | 真の値の TTree(sim)を書く ROOT ファイル(空なら書かない) |
| `Write_HA_Data` | bool | true | true |  | true: HA の raw ファイルを書く |
| `Write_Anode_Data` | bool | true | true |  | true: アノードの raw ファイルを書く |
| `Write_Cathode_Data` | bool | true | true |  | true: カソードの raw ファイルを書く |

`run_sim.py --raw-dir <dir>` は `output_dir` と `truth_file`(`<dir>/sim_truth.root`)を上書きします。

パラメータは設定ファイル `config/simulation/smile2p_flight_parma_thermal_neutron_d10_raw.json` の `response[RawWriter2p].parameters` に書きます
(書き方は `config/README.md`)。1 回だけ変えて試すときは `run_sim.py ... --set response.RawWriter2p.Write_HA_Data=false` で上書きします。
「既定値(C++)」は設定ファイルに書かなかったときの値です。

## 3. ほかのモジュールとのやり取り

受け取るもの:

| モジュール | 関数 |
| --- | --- |
| `EdepPickUp` | `EventID()`, `PrimaryEnergy()`, `PrimaryPos()`, `PrimaryDir()`, `ScintiEnergy()`, `TPCEnergy()`, `AntiEnergy()` |
| `TPCEmulator` | `NBoard()`, `EventTime()`, `Hits(side, board)`, `NHit(side, board)`, `FADC(side, board)` |
| `HAEmulator` | `ADC(haIndex)`, `HAName(haIndex)` |

ほかのモジュールに値を渡す関数はありません(チェーンの最後)。

真の値のツリー `sim`(raw のイベントと 1 対 1、同じ順):

| 分岐 | 単位 | 中身 |
| --- | --- | --- |
| `EventID` | | Geant4 のイベント番号(`edep` の `event_id`) |
| `SrcEng` | keV | 一次粒子のエネルギー |
| `SrcPos[3]`, `SrcDir[3]` | mm, なし | 一次粒子の位置、方向 |
| `E_psa`, `E_tpc`, `E_anti` | keV | シンチ・TPC・アンチのエネルギーの合計 |

## 4. 確かめる

In [1]:
import sys, json
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
import ROOT
ROOT.gROOT.SetBatch(True)

REPO = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "run_script" / "run_sim.py").exists())
sys.path.insert(0, str(REPO / "config"))
import ericanext_config as ec
CONFIG = "smile2p_flight_parma_thermal_neutron_d10_raw"
cfg = ec.load_sim(CONFIG)
# 1_DataIO/DataIO_example.ipynb の出力(d10 の熱中性子、100 万個、seed 1)を使う
SIM = REPO / "tutorials_simulation" / "cache" / "d10_seed1_1M"
EDEP, RAW = SIM / "edep.root", SIM / "raw"
assert EDEP.exists(), "先に ../../DataIO_example.ipynb を実行してください"

def flat(column):
    # RDataFrame.AsNumpy() の vector 型の列 -> 全イベントをつないだ 1 次元の numpy 配列
    return np.concatenate([np.asarray(v) for v in column]) if len(column) else np.array([])

In [2]:
files = sorted(RAW.glob("*.raw"))
for p in files:
    print(f"{p.name:28s} {p.stat().st_size:>10,d} bytes")
truth = ROOT.RDataFrame("sim", str(RAW / "sim_truth.root")).AsNumpy(["EventID", "E_tpc", "E_psa"])
edep = ROOT.RDataFrame("edep", str(EDEP)).AsNumpy(["event_id", "e_tpc", "e_scinti"])
idx = {int(k): i for i, k in enumerate(edep["event_id"])}
j = np.array([idx[int(k)] for k in truth["EventID"]])
print("sim の行:", len(j), " edep と同じ値:", np.allclose(edep["e_tpc"][j], truth["E_tpc"]) and np.allclose(edep["e_scinti"][j], truth["E_psa"]))
print("".join(l for l in open(SIM / "run_sim.log") if l.startswith("RawWriter2p:")))

HA01_00000.raw                   15,120 bytes
HA02_00000.raw                   15,120 bytes
HA03_00000.raw                   15,120 bytes
HA07_00000.raw                   15,120 bytes
HA08_00000.raw                   15,120 bytes
HA09_00000.raw                   15,120 bytes
HA13_00000.raw                   15,120 bytes
HA14_00000.raw                   15,120 bytes
HA15_00000.raw                   15,120 bytes
HA19_00000.raw                   15,120 bytes
HA20_00000.raw                   15,120 bytes
HA21_00000.raw                   15,120 bytes
HA25_00000.raw                   15,120 bytes
HA26_00000.raw                   15,120 bytes
HA27_00000.raw                   15,120 bytes
HA31_00000.raw                   15,120 bytes
HA32_00000.raw                   15,120 bytes
HA33_00000.raw                   15,120 bytes
uTPC_anode1_00000.raw         1,415,000 bytes
uTPC_anode2_00000.raw         1,452,560 bytes
uTPC_anode3_00000.raw         1,467,100 bytes
uTPC_cathode1_00000.raw       1,48

sim の行: 270  edep と同じ値: True
RawWriter2p: 270 events written to ../tutorials_simulation/cache/d10_seed1_1M/raw



- raw のファイルの名前と形は実データ(`uTPC_anode1_00000.raw` など)と同じなので、`ReadData` の `data_dir` にこのディレクトリを渡せば読めます。
- `sim` のイベント番号で `edep` の行を探すと、エネルギーの値が一致します。真の値と再構成の結果は行の順で対応します(`../../DataIO_example.ipynb` の 7 節)。